# Spark

## Installation de PySpark

In [ ]:
!pip install pyspark

## Imports

In [ ]:
import typing

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas
import pyspark
import pyspark.ml
import pyspark.sql.functions as F
import seaborn as sns

## Chargement des données

Pour ces travaux pratiques, nous allons utiliser des données de transactions immobilières sur la France entière, entre 2014 et 2022.

Ces données proviennent du site d'[open data français](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/). Nous utiliserons là une version retravaillée qui regroupe les transactions, qui provient du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) sur GitHub.

In [ ]:
!git clone https://github.com/mlambda/dataset-dvf.git

En Spark SQL, on commence toujours par définir une session, nommée par convention `spark`. Cette session permet de préciser comment accéder au cluster, quelques options, etc. Ici, nous l'utiliserons dans sa forme la plus simple.

In [ ]:
spark = (pyspark.sql.SparkSession.builder
                                 .appName("DVF")
                                 .getOrCreate())

*Vous pouvez procéder au chargement des données avec la fonction [`pyspark.sql.DataFrameReader.parquet`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.parquet.html). Vous pouvez appeler cette fonction avec la syntaxe `spark.read.parquet(...)` où `spark` est la session Spark que nous venons de créer et `...` vos arguments.*

In [ ]:
# Votre code ici

*Une fois les données disponibles dans une feuille de données Spark, créez une colonne `date` à partir des colonnes `annee_mutation`, `mois_mutation` & `jour_mutation` avec la fonction [`pyspark.sql.functions.make_date`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.make_date.html), puis supprimez ces colonnes pour économiser de la place en mémoire.*

In [ ]:
# Votre code ici

Enfin, affichez le type des colonnes de la feuille de données.

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = spark.read.parquet("dataset-dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df = (df.withColumn("date", F.make_date("annee_mutation",
                                        "mois_mutation",
                                        "jour_mutation"))
        .drop("annee_mutation", "mois_mutation", "jour_mutation"))
df.show(10)

In [ ]:
df.printSchema()

## Nombre de transactions par année

Pour commencer, nous allons créer le décompte par année du nombre de transactions (lignes du jeu de données).

*Utilisez les fonctions suivantes pour fournir les données à la fonction de création de diagramme :*
- *La fonction [`pyspark.sql.functions.year`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.year.html) pour exploiter la colonne `date`.*
- *La fonction [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) pour regrouper les données.*
- *La fonction [`pyspark.sql.DataFrame.toPandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html) pour retourner une structure de données facilement exploitable par seaborn.*

La fonction de création de diagramme attend les données dans une feuille de données Pandas organisée de cette manière :

```
   year    count
0  2014  1032967
1  2015  1128516
2  2016  1210587
3  2017  1369101
4  2018  1351856
5  2019  1431358
6  2020  1268193
7  2021  1537648
8  2022   508837
```

In [ ]:
def get_data_transactions_by_year() -> pandas.DataFrame:
  pass  # Votre code ici


def transactions_by_year() -> None:
  data = get_data_transactions_by_year()

  fig, ax = plt.subplots()
  sns.barplot(data, x="year", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Décompte des transactions par année")
  ax.set_xlabel("Année")
  ax.set_ylabel("Nombre de transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


# transactions_by_year()

### Solution

In [ ]:
def get_data_transactions_by_year() -> pandas.DataFrame:
  return (df.groupby(F.year("date").alias("year"))
            .count()
            .orderBy("year")
            .toPandas())


def transactions_by_year() -> None:
  data = get_data_transactions_by_year()

  fig, ax = plt.subplots()
  sns.barplot(data, x="year", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Décompte des transactions par année")
  ax.set_xlabel("Année")
  ax.set_ylabel("Nombre de transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


transactions_by_year()

## Valeur foncière totale par département

Dans cet exercice, nous allons regrouper nos données selon les valeurs d'une colonne (ici le département de la transaction) et afficher la valeur foncière totale échangée par groupe obtenu.

- *Regroupez les données par département à l'aide de la fonction [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html).*
- *Triez le résultat par ordre décroissant à l'aide de la fonction [`pyspark.sql.DataFrame.orderBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html).*
- *Convertissez le résultat en série Pandas (avec comme index les départements).*

La série Pandas attendue a la forme suivante :

```
code_departement
75    1.821853e+11
92    1.327795e+11
6     8.143718e+10
33    8.015320e+10
69    7.632324e+10
          ...     
55    1.998241e+09
15    1.907638e+09
52    1.753427e+09
23    1.248303e+09
48    8.985644e+08
Name: total, Length: 97, dtype: float64
```

In [ ]:
def get_data_value_by_department() -> pandas.Series:
  pass  # Votre code ici


def value_by_department() -> None:
  data = get_data_value_by_department()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Montant total des transactions par département")
  ax.set_xlabel("Département")
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(
      mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# value_by_department()

### Solution

In [ ]:
def get_data_value_by_department() -> pandas.Series:
  return (df.groupby("code_departement")
            .agg(F.sum("valeur_fonciere").alias("total"))
            .orderBy("total", ascending=False)
            .toPandas()
            .set_index("code_departement")
            .iloc[:, 0])


def value_by_department() -> None:
  data = get_data_value_by_department()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Montant total des transactions par département")
  ax.set_xlabel("Département")
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(
      mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_department()

## Valeur foncière totale par région et par an

Nous allons étudier dans cette partie comment décliner une variable catégorielle en fonction d'une autre variable : nous allons produire un diagramme de la valeur foncière totale changée par région, et cette valeur sera elle-même déclinée par année.

Pour cela, commençons par récupérer un jeu de données qui nous permettra de regrouper les départements en régions :

In [ ]:
!wget https://www.data.gouv.fr/fr/datasets/r/987227fb-dcb2-429e-96af-8979f97c9c84 -O regions.csv

La fonction suivante renvoie une feuille de données Spark qui permettra de facilement ajoindre la région à notre feuille de données de travail.

In [ ]:
def get_regions_df() -> pyspark.sql.DataFrame:
  return (spark.read.csv("regions.csv", inferSchema=True, header=True)
                    .drop("dep_name")
                    .withColumn("num_dep",
                                F.regexp_replace("num_dep", "^0+", "")))


get_regions_df().show(10)

- *Ajoutez une colonne qui contient le nom de région dans votre feuille de données de travail, grâce à la fonction [`pyspark.sql.DataFrame.join`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html).*
- *Renvoyez une feuille de données Pandas qui contient la somme de la valeur foncière échangée par région et par année à l'aide de la fonction [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html).*
- *Renvoyez aussi une série Pandas : les noms de région triés, de la région qui a la somme de valeur foncière échangée la plus importante à la moins importante.*

La feuille de données à renvoyer a la forme suivante :

```
                 region_name  year         total
0              Île-de-France  2021  9.168523e+10
1    Bourgogne-Franche-Comté  2021  8.240557e+09
2                 La Réunion  2021  1.681132e+09
3                     Guyane  2021  5.290445e+08
4                   Bretagne  2021  1.613739e+10
..                       ...   ...           ...
148          Hauts-de-France  2014  1.532841e+10
149            Île-de-France  2014  5.324466e+10
150                Normandie  2022  2.027011e+09
151  Bourgogne-Franche-Comté  2014  4.475983e+09
152                 Bretagne  2014  6.876292e+09
```

La série à renvoyer a la forme suivante :

```
0                  Île-de-France
1           Auvergne-Rhône-Alpes
2     Provence-Alpes-Côte d'Azur
3             Nouvelle-Aquitaine
4                      Occitanie
5                Hauts-de-France
6               Pays de la Loire
7                       Bretagne
8                      Normandie
9            Centre-Val de Loire
10       Bourgogne-Franche-Comté
11                     Grand Est
12                    La Réunion
13                         Corse
14                    Guadeloupe
15                    Martinique
16                        Guyane
Name: region_name, dtype: object
```

In [ ]:
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame,
                                                pandas.Series]:
  pass  # Votre code ici


def value_by_region_by_year() -> None:
  data, sorted_index = get_data_value_by_region_by_year()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data,
              x="region_name",
              y="total",
              hue="year",
              order=sorted_index,
              ax=ax)
  sns.despine(fig)
  ax.set_title("Montant total des transactions par département et par année")
  ax.set_xlabel("Région")
  plt.xticks(rotation=60)
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(
      mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# value_by_region_by_year()

### Solution

In [ ]:
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame,
                                                pandas.Series]:
  regions_df = get_regions_df()
  joined = (df.join(regions_df, df.code_departement == regions_df.num_dep)
              .drop("num_dep"))
  data = (joined.groupBy(["region_name", F.year("date").alias("year")])
                .agg(F.sum("valeur_fonciere").alias("total"))
                .toPandas())
  sorted_index = (joined.groupBy("region_name")
                        .agg(F.sum("valeur_fonciere").alias("total"))
                        .orderBy("total", ascending=False)
                        .select("region_name")
                        .toPandas()
                        .iloc[:, 0])
  return data, sorted_index


def value_by_region_by_year() -> None:
  data, sorted_index = get_data_value_by_region_by_year()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data,
              x="region_name",
              y="total",
              hue="year",
              order=sorted_index,
              ax=ax)
  sns.despine(fig)
  ax.set_title("Montant total des transactions par département et par année")
  ax.set_xlabel("Région")
  plt.xticks(rotation=60)
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(
      mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_region_by_year()

## Prix des maisons et des appartements

Dans cet exercice, nous allons calculer le prix au mètre carré des appartements et maisons séparément et afficher l'évolution de ces prix dans l'intervalle considéré.

- *Pour les appartements, filtrez la feuille de calcul de travail pour ne conserver que les lignes où :*
    - *`nombre_appartements`, `valeur_fonciere` & `surface_reelle_bati_appartements` sont strictement positifs.*
    - *`nombre_maisons` vaut `0`.*
- *De la même manière, pour les maisons, filtrez la feuille de calcul de travail pour ne conserver que les lignes où :*
    - *`nombre_maisons`, `valeur_fonciere` & `surface_reelle_bati_maisons` sont strictement positifs.*
    - *`nombre_appartements` vaut `0`.*
- *Utilisez ces données filtrées pour calculer le prix au mètre carré pour chaque mois disponible dans le jeu de données, pour les maisons et les appartements séparément. Vous pourrez utiliser les fonctions suivantes :*
    - La fonction [`pyspark.sql.functions.date_format`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html) pour obtenir une colonne qui contient la date sous la forme `yyyy-MM`.
    - La fonction [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
- *La fonction de diagramme utilisée, [`seaborn.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html), attend des données en forme longue. Utilisez la fonction [`pyspark.sql.DataFrame.melt`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.melt.html) pour les lui fournir.*

La forme de la feuille de données Pandas attendue est la suivante :

```
       month       type        price
0    2014-01  apartment  3005.224170
1    2014-01      house  1588.628790
2    2014-02  apartment  2878.714589
3    2014-02      house  1622.620962
4    2014-03  apartment  2911.870507
..       ...        ...          ...
199  2022-04      house  2190.928353
200  2022-05  apartment  4164.208224
201  2022-05      house  2150.579262
202  2022-06  apartment  4530.162208
203  2022-06      house  2242.808145
```

In [ ]:
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  pass  # Votre code ici


def houses_apartments_prices() -> None:
  data = get_data_houses_apartments_prices()

  fig, ax = plt.subplots()
  sns.lineplot(data, x="month", y="price", hue="type", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Valeur moyenne")
  ax.set_title("Prix au mètre carré")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(
      mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


# houses_apartments_prices()

### Solution

In [ ]:
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  mask_apartments = ((df.nombre_appartements > 0)
                     & (df.nombre_maisons == 0)
                     & (df.valeur_fonciere > 0)
                     & (df.surface_reelle_bati_appartements > 0))
  mask_houses = ((df.nombre_appartements == 0)
                 & (df.nombre_maisons > 0)
                 & (df.valeur_fonciere > 0)
                 & (df.surface_reelle_bati_maisons > 0))

  def get_prices(mask: typing.Any, area_column: str, column_name: str
                 ) -> pyspark.sql.DataFrame:
    return (df.filter(mask)
              .groupby(F.date_format("date", "yyyy-MM").alias("month"))
              .agg(F.sum("valeur_fonciere"), F.sum(area_column))
              .withColumn(column_name, F.col("sum(valeur_fonciere)")
                                       / F.col(f"sum({area_column})"))
              .drop("sum(valeur_fonciere)", f"sum({area_column})"))

  prices_apartments = get_prices(mask_apartments,
                                 "surface_reelle_bati_appartements",
                                 "apartment")
  prices_houses = get_prices(mask_houses,
                             "surface_reelle_bati_maisons",
                             "house")

  return (prices_apartments.join(prices_houses, "month")
                           .orderBy("month")
                           .melt(ids="month",
                                 values=["apartment", "house"],
                                 variableColumnName="type",
                                 valueColumnName="price")
                           .toPandas())


def houses_apartments_prices() -> None:
  data = get_data_houses_apartments_prices()

  fig, ax = plt.subplots()
  sns.lineplot(data, x="month", y="price", hue="type", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Valeur moyenne")
  ax.set_title("Prix au mètre carré")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(
      mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


houses_apartments_prices()

## Corrélations entre variables

Nous allons pour finir voir comment calculer les corrélations linéaires avec la valeur foncière et plus généralement entre variables en Spark. Le procédé est beaucoup plus complexe qu'en Pandas, mais il nous permettra d'observer le fonctionnement de la MLLib.

In [ ]:
def get_data_corrs() -> pandas.DataFrame:
  vector_col = "corr_features"

  corr_columns = [f.name
                  for f in df.schema.fields
                  if isinstance(f.dataType, pyspark.sql.types.NumericType)]

  assembler = pyspark.ml.feature.VectorAssembler(inputCols=corr_columns,
                                                outputCol=vector_col)
  df_vector = assembler.transform(df).select(vector_col)
  matrix = pyspark.ml.stat.Correlation.corr(df_vector, vector_col)
  values = matrix.collect()[0][f"pearson({vector_col})"].values

  result = pandas.DataFrame(values.reshape(-1, len(corr_columns)),
                            columns=corr_columns,
                            index=corr_columns)
  index = result.valeur_fonciere.sort_values(ascending=False, key=abs).index
  return result.loc[index, index]


def corrs() -> None:
  data = get_data_corrs()
  print(data)
  fig, ax = plt.subplots(figsize=(15, 10))
  sns.heatmap(data, vmin=-1, vmax=1, cmap="vlag", ax=ax)
  ax.set_title("Corrélations entre variables triées par corrélation au prix")
  fig.show()


corrs()